# 第6章：Transformer 架构全景

> **预计学习时间：约 30 分钟**
>
> 上一章我们学了注意力机制——Transformer 的核心引擎。现在是时候把所有组件组装成一台完整的机器了。本章将带你从零理解 Transformer 的完整结构，亲手实现 Transformer Block，并探索 BERT、GPT-2、T5 三种主流变体的区别。

**运行环境**: Kaggle Notebook (CPU 即可，加载预训练模型需要安装 transformers 库)
**前置要求**: 完成第5章注意力机制

---
## 0. 环境准备

In [ ]:
# =============================================
# 0. 安装并导入依赖
# =============================================

# 安装 transformers 库，用于加载预训练模型（BERT、GPT-2 等）
# -q 表示安静模式，减少输出信息
!pip install -q transformers

import torch
import torch.nn as nn
import torch.nn.functional as F  # 包含 softmax、relu、gelu 等常用激活函数
import numpy as np
import matplotlib.pyplot as plt
import math  # 用于 sqrt 计算缩放因子
import time  # 用于计时对比实验

# transformers 库：Hugging Face 提供的预训练模型库
# AutoModel: 自动加载模型结构
# AutoTokenizer: 自动加载对应的分词器
from transformers import AutoModel, AutoTokenizer

# 设置随机种子，确保每次运行结果一致
torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.size'] = 12

print(f"PyTorch 版本: {torch.__version__}")
print("本章主要使用 CPU，无需 GPU")

---
## 1. Transformer 的整体结构

### 类比：汽车工厂的流水线

想象一家汽车工厂。原材料（钢板、轮胎）进入流水线后，要经过一连串的工位——冲压、焊接、喷漆、组装——最后变成一辆完整的汽车。

Transformer 就是一条**文本处理流水线**：

- **原材料** = 文本（一串词）
- **工位** = Transformer Block（每个 Block 做同样的加工：注意力 + 前馈网络）
- **成品** = 对每个词的理解向量（可以用来做分类、翻译、生成等任务）

关键特点：每个工位的结构完全相同，但里面的工人（权重参数）不同——它们各自学会了关注文本的不同方面。

### Transformer 的完整流程

```
Input Text: "I love deep learning"
   |
   v
[Tokenizer] -------- 把文字切成 token（词/子词）
   |
   v
[Embedding] -------- 每个 token 变成一个向量（第4章）
   +
[Positional Encoding] -- 加入位置信息（第7章详解）
   |
   v
[Transformer Block 1] -- 第一次加工：提取基础特征
   |
   v
[Transformer Block 2] -- 第二次加工：提取更高层特征
   |
   v
  ...                   （重复 N 次，GPT-2 有 12 层）
   |
   v
[Transformer Block N] -- 第 N 次加工：提取最高层特征
   |
   v
[Output Layer] --------- 生成最终结果（下一个词的概率）
```

![Transformer 整体架构](images/transformer_architecture.png)

*图1：Transformer 的整体架构——从输入文本到输出结果的完整流水线*

> 💡 **记忆要点**
> - Transformer = 流水线：输入 → Embedding + 位置编码 → N 个 Block → 输出
> - 每个 Block 结构相同，但参数不同
> - 层数越多，理解越深（但也越慢、越耗内存）

---
## 2. Transformer Block 的内部结构

### 每个 Block 里面有什么？

回到工厂类比：每个工位内部也有固定的流程。Transformer Block 内部有**四个核心组件**，按固定顺序执行：

| 步骤 | 组件 | 作用 | 类比 |
|------|------|------|------|
| 1 | Multi-Head Attention | 让每个词关注其他词，收集上下文信息 | 多个检查员同时从不同角度检查产品 |
| 2 | Add & LayerNorm | 残差连接 + 层归一化，稳定训练 | 每次加工后做质量检查，确保不跑偏 |
| 3 | Feed-Forward Network (FFN) | 对每个词独立做非线性变换 | 对产品做精加工 |
| 4 | Add & LayerNorm | 再次残差连接 + 层归一化 | 再次质量检查 |

### 数据在 Block 内部的流动

```
输入 x [batch, seq_len, d_model]
  |
  +-------> Multi-Head Attention(x, x, x)
  |                   |
  +--- 残差连接 ----> Add  (x + Attention输出)
                      |
                  LayerNorm
                      |
  +-------> Feed-Forward Network
  |                   |
  +--- 残差连接 ----> Add  (x + FFN输出)
                      |
                  LayerNorm
                      |
                    输出 [batch, seq_len, d_model]
```

![Transformer Block 内部结构](images/transformer_block.png)

*图2：Transformer Block 的内部结构——注意力 + 残差 + FFN + 残差*

### 为什么需要残差连接？

残差连接（Residual Connection）就是把输入直接加到输出上：`output = x + f(x)`

想象你在工厂流水线上加工一个零件。残差连接就像是说："加工完之后，把原始零件和加工结果放在一起传给下一站。" 这样即使某次加工效果不好，原始信息也不会丢失。

### 为什么需要 LayerNorm？

LayerNorm 把每个向量的数值规范化到均值为 0、方差为 1 的范围。这就像质量检查：确保每一步的输出都在合理的数值范围内，防止数值爆炸或消失。

### FFN 做了什么？

FFN 是两层全连接网络：先把维度扩大 4 倍（d_model → 4×d_model），过一个激活函数（GELU），再缩回来（4×d_model → d_model）。

Attention 负责"词与词之间的交流"，FFN 负责"每个词独立地消化和处理信息"。

> 💡 **记忆要点**
> - Block = Attention + Add&Norm + FFN + Add&Norm
> - 残差连接：防止信息丢失，让深层网络能训练
> - LayerNorm：稳定数值范围
> - FFN：先扩展 4 倍再压缩回来，让每个词独立处理信息

In [ ]:
# =============================================
# 2.1 从零实现 Transformer Block
# =============================================

class MultiHeadAttention(nn.Module):
    """多头注意力模块
    
    把 d_model 维度拆分成 n_heads 个头，每个头独立做注意力，
    最后拼接结果。这样模型能同时关注不同类型的关系。
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        # d_model 必须能被 n_heads 整除
        # 例如 d_model=512, n_heads=8 → 每个头处理 64 维
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 每个头的维度
        
        # 四个线性变换：Q、K、V 的投影 + 最终的输出投影
        # 用一个大矩阵同时计算所有头的 Q/K/V，效率更高
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # 输出投影
    
    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len, d_model]
        mask: [batch_size, 1, 1, seq_len] 或 [batch_size, 1, seq_len, seq_len]
        """
        batch_size, seq_len, _ = x.shape
        
        # Step 1: 线性变换得到 Q, K, V
        # [batch, seq_len, d_model] → [batch, seq_len, d_model]
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Step 2: 拆分成多个头
        # [batch, seq_len, d_model] → [batch, seq_len, n_heads, d_k] → [batch, n_heads, seq_len, d_k]
        # view: 改变张量形状（不改变数据）
        # transpose(1,2): 交换第1维和第2维，把 n_heads 提前
        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Step 3: Scaled Dot-Product Attention（和第5章一样的公式）
        # scores: [batch, n_heads, seq_len, seq_len]
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        
        # 如果有 mask，把被屏蔽的位置设为负无穷
        # softmax(-inf) = 0，所以这些位置的注意力权重为 0
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # softmax 归一化，得到注意力权重
        attn_weights = F.softmax(scores, dim=-1)
        
        # 加权求和 Value
        # [batch, n_heads, seq_len, seq_len] @ [batch, n_heads, seq_len, d_k]
        # → [batch, n_heads, seq_len, d_k]
        context = attn_weights @ V
        
        # Step 4: 拼接所有头
        # [batch, n_heads, seq_len, d_k] → [batch, seq_len, n_heads, d_k] → [batch, seq_len, d_model]
        # contiguous(): 确保内存连续（transpose 后内存可能不连续）
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Step 5: 最终线性投影
        # 把拼接后的多头结果映射回 d_model 维度
        output = self.W_O(context)
        
        return output, attn_weights


class FeedForward(nn.Module):
    """前馈网络（Position-wise Feed-Forward Network）
    
    两层全连接：先扩展到 4*d_model，再压缩回 d_model
    中间用 GELU 激活函数（比 ReLU 更平滑，GPT-2 和 BERT 都使用 GELU）
    
    为什么要先扩展再压缩？
    → 更大的中间维度给模型更多的"思考空间"来处理信息
    → 类似于人做题时先在草稿纸上展开计算，再把答案写到答题纸上
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        # d_ff 默认是 d_model 的 4 倍（Transformer 原始论文的设置）
        if d_ff is None:
            d_ff = 4 * d_model
        # 第一层：扩展维度 d_model → d_ff
        self.fc1 = nn.Linear(d_model, d_ff)
        # 第二层：压缩回来 d_ff → d_model
        self.fc2 = nn.Linear(d_ff, d_model)
        # GELU 激活函数：Gaussian Error Linear Unit
        self.activation = nn.GELU()
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # → [batch, seq_len, d_ff] → [batch, seq_len, d_model]
        return self.fc2(self.activation(self.fc1(x)))


class TransformerBlock(nn.Module):
    """一个完整的 Transformer Block
    
    结构：
      x → MultiHeadAttention → Add & LayerNorm → FFN → Add & LayerNorm → output
    
    这是 Transformer 最核心的组件，整个模型就是多个 Block 堆叠而成
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        # LayerNorm：对每个样本的特征维度做归一化
        # 和 BatchNorm 不同，LayerNorm 不依赖 batch 中的其他样本
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # Dropout：训练时随机丢弃一部分连接，防止过拟合
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Step 1: Multi-Head Attention + 残差连接 + LayerNorm
        # 残差连接：x + Attention(x) → 即使 Attention 学不好，原始信息也能传过去
        attn_output, attn_weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))  # Add & Norm
        
        # Step 2: FFN + 残差连接 + LayerNorm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))   # Add & Norm
        
        return x, attn_weights


# 测试 TransformerBlock
d_model = 64    # 嵌入维度（实际 GPT-2 用 768，这里用小值便于观察）
n_heads = 4     # 注意力头数（实际 GPT-2 用 12）
batch_size = 2
seq_len = 10

block = TransformerBlock(d_model=d_model, n_heads=n_heads)
x = torch.randn(batch_size, seq_len, d_model)

output, weights = block(x)

print("Transformer Block 测试:")
print(f"  输入形状:  {x.shape}  [batch={batch_size}, seq_len={seq_len}, d_model={d_model}]")
print(f"  输出形状:  {output.shape}  [形状不变！]")
print(f"  注意力权重: {weights.shape}  [batch, n_heads={n_heads}, seq_len, seq_len]")

# 统计参数量
total_params = sum(p.numel() for p in block.parameters())
print(f"\n参数量统计:")
print(f"  Multi-Head Attention: 4 × {d_model}×{d_model} = {4 * d_model**2:,}")
print(f"  FFN: {d_model}×{4*d_model} + {4*d_model}×{d_model} + 偏置 = {d_model*4*d_model + 4*d_model*d_model + 4*d_model + d_model:,}")
print(f"  LayerNorm: 2 × 2 × {d_model} = {4*d_model}")
print(f"  总计: {total_params:,}")

In [ ]:
# =============================================
# 2.2 追踪张量形状变化
# =============================================
# 理解每一步的形状变化对调试 Transformer 非常重要

d_model = 64
n_heads = 4
d_k = d_model // n_heads  # 64 // 4 = 16，每个头处理 16 维
d_ff = 4 * d_model         # 256，FFN 中间层维度
batch_size = 2
seq_len = 8

x = torch.randn(batch_size, seq_len, d_model)
print("=" * 60)
print("Transformer Block 内部张量形状追踪")
print("=" * 60)

print(f"\n输入 x: {list(x.shape)}")
print(f"  含义: [batch={batch_size}, seq_len={seq_len}, d_model={d_model}]")

# --- Multi-Head Attention ---
print(f"\n--- Multi-Head Attention ---")

# 线性投影
W_Q = nn.Linear(d_model, d_model, bias=False)
Q = W_Q(x)
print(f"  Q = W_Q(x): {list(Q.shape)}  [batch, seq_len, d_model]")

# 拆分成多个头
Q_heads = Q.view(batch_size, seq_len, n_heads, d_k).transpose(1, 2)
print(f"  Q 拆分成 {n_heads} 个头: {list(Q_heads.shape)}  [batch, n_heads, seq_len, d_k={d_k}]")

# 注意力分数
K_heads = torch.randn_like(Q_heads)
scores = Q_heads @ K_heads.transpose(-2, -1) / math.sqrt(d_k)
print(f"  注意力分数 Q@K^T/sqrt(d_k): {list(scores.shape)}  [batch, n_heads, seq_len, seq_len]")

# softmax
attn = F.softmax(scores, dim=-1)
print(f"  Softmax 归一化: {list(attn.shape)}  [每行和为 1]")

# 加权求和
V_heads = torch.randn(batch_size, n_heads, seq_len, d_k)
context = attn @ V_heads
print(f"  加权求和 attn@V: {list(context.shape)}  [batch, n_heads, seq_len, d_k]")

# 拼接多头
concat = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
print(f"  拼接多头: {list(concat.shape)}  [batch, seq_len, d_model]  ← 回到原始形状！")

# --- 残差 + LayerNorm ---
print(f"\n--- Add & LayerNorm ---")
norm1 = nn.LayerNorm(d_model)
after_norm = norm1(x + concat)
print(f"  x + Attention(x): {list(after_norm.shape)}  [残差连接保持形状不变]")

# --- FFN ---
print(f"\n--- Feed-Forward Network ---")
fc1 = nn.Linear(d_model, d_ff)
hidden = F.gelu(fc1(after_norm))
print(f"  扩展: {list(hidden.shape)}  [d_model={d_model} → d_ff={d_ff}] (扩大 4 倍!)")

fc2 = nn.Linear(d_ff, d_model)
ffn_out = fc2(hidden)
print(f"  压缩: {list(ffn_out.shape)}  [d_ff={d_ff} → d_model={d_model}] (压回原尺寸)")

# --- 最终残差 + LayerNorm ---
print(f"\n--- 最终 Add & LayerNorm ---")
norm2 = nn.LayerNorm(d_model)
final = norm2(after_norm + ffn_out)
print(f"  最终输出: {list(final.shape)}  [和输入形状完全一致!]")

print(f"\n{'=' * 60}")
print(f"关键结论: 输入和输出形状完全一致 {list(x.shape)} → {list(final.shape)}")
print(f"所以 Transformer Block 可以无限堆叠!")

---
## 3. Encoder vs Decoder：有什么区别？

Transformer 原始论文提出了 Encoder-Decoder 架构（用于机器翻译）。后来人们发现，只用 Encoder 或只用 Decoder 也能做很多任务。

### 三种模式

| 特性 | Encoder | Decoder |
|------|---------|--------|
| 能看到哪些词？ | **所有词**（双向） | **只能看到前面的词**（单向） |
| Mask | 无 mask（或只 mask padding） | 因果 mask（下三角矩阵） |
| 典型用途 | 理解文本（分类、NER） | 生成文本（写作、对话） |
| 代表模型 | BERT | GPT |

### 核心区别：双向 vs 单向

```
Encoder（双向注意力）:
  处理 "cat" 时，可以看到所有词:
  "The" ✓  "cat" ✓  "sat" ✓  "on" ✓  "the" ✓  "mat" ✓
  → 适合理解任务（知道完整上下文）

Decoder（单向注意力）:
  处理 "cat" 时，只能看到前面的词:
  "The" ✓  "cat" ✓  "sat" ✗  "on" ✗  "the" ✗  "mat" ✗
  → 适合生成任务（预测下一个词）
```

![Encoder vs Decoder 对比](images/encoder_decoder_variants.png)

*图3：Encoder（双向）vs Decoder（单向）——注意力 mask 的区别*

In [ ]:
# =============================================
# 3.1 实现 Encoder Block 和 Decoder Block
# =============================================
# 它们的结构几乎一样，唯一的区别是 Decoder 使用因果遮罩

def create_causal_mask(seq_len):
    """创建因果遮罩（下三角矩阵）
    
    1 = 可以看到, 0 = 被屏蔽
    对角线及以下为 1（可以看到自己和前面的词）
    对角线以上为 0（不能偷看后面的词）
    """
    # torch.tril: 取矩阵的下三角部分
    mask = torch.tril(torch.ones(seq_len, seq_len))
    # 增加 batch 和 head 维度: [1, 1, seq_len, seq_len]
    # 这样可以自动广播到任意 batch_size 和 n_heads
    return mask.unsqueeze(0).unsqueeze(0)


class EncoderBlock(nn.Module):
    """Encoder Block：双向注意力，没有 mask
    
    每个词可以看到序列中的所有其他词（包括前后）
    用于 BERT 等理解型模型
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.block = TransformerBlock(d_model, n_heads)
    
    def forward(self, x):
        # 不传 mask → 双向注意力（每个词看到所有词）
        return self.block(x, mask=None)


class DecoderBlock(nn.Module):
    """Decoder Block：单向注意力，使用因果 mask
    
    每个词只能看到自己和前面的词，不能偷看未来
    用于 GPT 等生成型模型
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.block = TransformerBlock(d_model, n_heads)
    
    def forward(self, x):
        seq_len = x.size(1)
        # 创建因果遮罩，阻止看到未来的词
        mask = create_causal_mask(seq_len).to(x.device)
        return self.block(x, mask=mask)


# 对比 Encoder 和 Decoder 的注意力模式
d_model = 64
n_heads = 4
seq_len = 6
x = torch.randn(1, seq_len, d_model)

torch.manual_seed(42)
encoder = EncoderBlock(d_model, n_heads)
enc_out, enc_weights = encoder(x)

torch.manual_seed(42)
decoder = DecoderBlock(d_model, n_heads)
dec_out, dec_weights = decoder(x)

# 可视化对比
words = ["The", "cat", "sat", "on", "the", "mat"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, w, title in zip(
    axes,
    # [0, 0]: 取第一个样本的第一个注意力头
    [enc_weights[0, 0].detach().numpy(), dec_weights[0, 0].detach().numpy()],
    ['Encoder: Bidirectional (No Mask)', 'Decoder: Causal Mask (Left-to-Right)']
):
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
    ax.set_xticks(range(len(words)))
    ax.set_yticks(range(len(words)))
    ax.set_xticklabels(words, fontsize=11)
    ax.set_yticklabels(words, fontsize=11)
    ax.set_xlabel('Key (attended to)', fontsize=11)
    ax.set_ylabel('Query (attending)', fontsize=11)
    ax.set_title(title, fontsize=13)
    for i in range(len(words)):
        for j in range(len(words)):
            color = 'white' if w[i, j] > 0.25 else 'black'
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=10, color=color)

plt.tight_layout()
plt.show()

print("左图（Encoder）: 每个词可以看到所有词 → 适合理解任务（如 BERT）")
print("右图（Decoder）: 右上角为 0 — 未来信息被完全屏蔽 → 适合生成任务（如 GPT）")
print(f"\n注意：模型结构完全一样! 唯一的区别就是有没有因果 mask")

---
## 4. 三种 Transformer 变体

根据使用 Encoder、Decoder 还是两者兼有，Transformer 分为三种变体：

| 变体 | 结构 | 代表模型 | 适合的任务 | 类比 |
|------|------|---------|-----------|------|
| Encoder-only | 只有 Encoder | BERT | 文本分类、情感分析、命名实体识别 | 阅读理解：读完全文再答题 |
| Decoder-only | 只有 Decoder | GPT-2, GPT-4, LLaMA | 文本生成、对话、代码生成 | 写作文：从左到右逐字写 |
| Encoder-Decoder | 两者兼有 | T5, BART | 翻译、摘要、问答 | 口译：先听完再翻译 |

### 为什么不同任务需要不同变体？

**理解任务**（分类、识别）需要看到完整上下文才能做出判断 → Encoder（双向）

**生成任务**（写文章、聊天）需要一个词一个词地输出 → Decoder（单向）

**转换任务**（翻译、摘要）需要先理解输入再生成输出 → Encoder-Decoder

```
Encoder-only (BERT):    输入 → [理解] → 分类标签
Decoder-only (GPT):     输入 → [生成] → 输出文本
Encoder-Decoder (T5):   输入 → [理解] → [生成] → 输出文本
```

### 现代趋势：Decoder-only 模型一统天下

虽然三种变体各有优势，但现在最成功的大模型几乎都是 **Decoder-only** 架构（GPT-4、Claude、LLaMA、DeepSeek）。原因是：
1. 生成能力最通用——任何任务都可以转化为"给定输入，生成输出"
2. 训练效率高——自回归目标函数简单有效
3. 规模效应好——模型越大，性能提升越明显

In [ ]:
# =============================================
# 4.1 加载真实模型：BERT 和 GPT-2
# =============================================
# 用 Hugging Face 的 transformers 库加载预训练模型
# 这两个都是 "小" 模型（约 1 亿参数），可以在 CPU 上运行

print("正在加载 BERT 和 GPT-2 模型（首次运行会下载模型文件）...")
print()

# 加载 BERT（Encoder-only）
# bert-base-uncased: 12 层, 768 维, 12 头, 110M 参数
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 加载 GPT-2（Decoder-only）
# gpt2: 12 层, 768 维, 12 头, 117M 参数
gpt2_model = AutoModel.from_pretrained('gpt2')
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

print("=" * 60)
print("BERT vs GPT-2 架构对比")
print("=" * 60)

# 统计参数量的辅助函数
# model.parameters() 返回所有参数张量的迭代器
# p.numel() 返回张量中元素的个数
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print(f"\nBERT (Encoder-only):")
print(f"  总参数量: {count_params(bert_model):,}")
print(f"  架构: 12 层 Encoder Block")
print(f"  注意力: 双向（每个词看到所有词）")

print(f"\nGPT-2 (Decoder-only):")
print(f"  总参数量: {count_params(gpt2_model):,}")
print(f"  架构: 12 层 Decoder Block")
print(f"  注意力: 单向（只看到前面的词）")

In [ ]:
# =============================================
# 4.2 查看模型内部结构
# =============================================
# 打印模型结构，对比 Encoder-only 和 Decoder-only 的差异

print("=" * 60)
print("BERT 模型结构（前 2 层）")
print("=" * 60)
# model.named_children() 返回模型的直接子模块
for name, module in bert_model.named_children():
    print(f"\n{name}:")
    if hasattr(module, 'layer'):
        # 只显示前 2 层，避免输出太长
        for i, layer in enumerate(module.layer[:2]):
            print(f"  Layer {i}: {layer.__class__.__name__}")
            for sub_name, sub_module in layer.named_children():
                print(f"    {sub_name}: {sub_module.__class__.__name__}")
        print(f"  ... (共 {len(module.layer)} 层)")
    else:
        print(f"  {module.__class__.__name__}")

print(f"\n{'=' * 60}")
print("GPT-2 模型结构（前 2 层）")
print("=" * 60)
for name, module in gpt2_model.named_children():
    print(f"\n{name}:")
    if hasattr(module, '__len__'):
        for i, layer in enumerate(list(module)[:2]):
            print(f"  Block {i}: {layer.__class__.__name__}")
            for sub_name, sub_module in layer.named_children():
                print(f"    {sub_name}: {sub_module.__class__.__name__}")
        if len(list(module)) > 2:
            print(f"  ... (共 {len(list(module))} 层)")
    else:
        print(f"  {module.__class__.__name__}")

print(f"\n观察: 两个模型的 Block 内部结构几乎一样")
print(f"  都有: Attention → LayerNorm → FFN → LayerNorm")
print(f"  区别: GPT-2 在 Attention 中使用了因果 mask")

In [ ]:
# =============================================
# 4.3 对比各组件参数量
# =============================================

def count_params_by_component(model, model_name):
    """按组件类型统计参数量
    
    遍历 model.named_parameters()，根据参数名判断属于哪个组件
    named_parameters() 返回 (参数名, 参数张量) 的迭代器
    参数名的格式类似 'encoder.layer.0.attention.self.query.weight'
    """
    components = {
        'Embedding': 0,
        'Attention': 0,
        'FFN': 0,
        'LayerNorm': 0,
        'Other': 0
    }
    
    for name, param in model.named_parameters():
        name_lower = name.lower()
        n = param.numel()  # numel() 返回张量中元素的总数
        if 'embed' in name_lower or 'wte' in name_lower or 'wpe' in name_lower:
            # wte: word token embedding, wpe: word position embedding (GPT-2)
            components['Embedding'] += n
        elif 'attn' in name_lower or 'attention' in name_lower:
            components['Attention'] += n
        elif 'mlp' in name_lower or 'intermediate' in name_lower or 'output.dense' in name_lower:
            # mlp: GPT-2 的 FFN, intermediate/output.dense: BERT 的 FFN
            components['FFN'] += n
        elif 'norm' in name_lower or 'ln' in name_lower:
            components['LayerNorm'] += n
        else:
            components['Other'] += n
    
    return components

bert_components = count_params_by_component(bert_model, 'BERT')
gpt2_components = count_params_by_component(gpt2_model, 'GPT-2')

# 打印对比表格
print(f"{'组件':<15} {'BERT':>15} {'GPT-2':>15}")
print("-" * 45)
for comp in ['Embedding', 'Attention', 'FFN', 'LayerNorm', 'Other']:
    b = bert_components[comp]
    g = gpt2_components[comp]
    print(f"{comp:<15} {b:>12,} {g:>15,}")
print("-" * 45)
print(f"{'Total':<15} {sum(bert_components.values()):>12,} {sum(gpt2_components.values()):>15,}")

---
## 5. 数据如何流过 Transformer

让我们用一个真实的例子，追踪一个句子从输入到输出经过 GPT-2 的完整过程。

### 完整的数据流

```
"I love deep learning"
        |
        v
[Tokenizer] ----→ token IDs: [40, 1842, 2769, 4673]    形状: [4]
        |
        v
[Token Embedding] --→ 查表得到词向量                     形状: [4, 768]
        +
[Position Embedding] → 加上位置编码                      形状: [4, 768]
        |
        v
[Transformer Block 1] → 第一层处理                       形状: [4, 768]
[Transformer Block 2] → 第二层处理                       形状: [4, 768]
       ...             → ...                             形状: [4, 768]
[Transformer Block 12] → 最后一层处理                    形状: [4, 768]
        |
        v
[LayerNorm] -------→ 最终归一化                          形状: [4, 768]
        |
        v
[LM Head] ---------→ 映射到词表大小                      形状: [4, 50257]
        |
        v
每个位置输出词表中每个词的概率
→ 取最后一个位置的最高概率 = 预测的下一个词
```

![数据在 Transformer 中的流动](images/data_flow.png)

*图4：数据流过 GPT-2 的每一步，注意形状始终保持 [seq_len, 768]（除了最后的 LM Head）*

> 💡 **关键洞察**: Transformer Block 不改变张量的形状！输入 [batch, seq_len, d_model]，输出还是 [batch, seq_len, d_model]。这就是为什么 Block 可以无限堆叠——像乐高积木一样。

In [ ]:
# =============================================
# 5.1 追踪数据流过 GPT-2
# =============================================
# 一步一步展示句子经过 GPT-2 的每个阶段

sentence = "I love deep learning"
print(f"输入句子: \"{sentence}\"")
print("=" * 60)

# Step 1: Tokenizer — 把文字切成 token
# return_tensors='pt' 表示返回 PyTorch 张量
tokens = gpt2_tokenizer(sentence, return_tensors='pt')
input_ids = tokens['input_ids']  # token ID 序列

print(f"\nStep 1: Tokenizer")
print(f"  Token IDs: {input_ids[0].tolist()}")
# convert_ids_to_tokens: 把 ID 转回可读的 token 字符串
token_strs = gpt2_tokenizer.convert_ids_to_tokens(input_ids[0])
print(f"  Tokens:    {token_strs}")
print(f"  形状:      {list(input_ids.shape)}  [batch=1, seq_len={input_ids.shape[1]}]")

# Step 2: Embedding — 查表得到词向量
# gpt2_model.wte: Word Token Embedding 层
# 本质是一个大查找表: vocab_size × d_model = 50257 × 768
token_embeds = gpt2_model.wte(input_ids)
print(f"\nStep 2: Token Embedding")
print(f"  查找表大小: {list(gpt2_model.wte.weight.shape)}  [vocab_size=50257, d_model=768]")
print(f"  输出形状: {list(token_embeds.shape)}  [batch, seq_len, d_model=768]")

# Step 3: Position Embedding — 加入位置信息
# gpt2_model.wpe: Word Position Embedding 层
# 每个位置有一个固定的位置向量（也是可学习的）
seq_len = input_ids.shape[1]
# torch.arange(seq_len) 生成 [0, 1, 2, ..., seq_len-1]
position_ids = torch.arange(seq_len).unsqueeze(0)  # [1, seq_len]
position_embeds = gpt2_model.wpe(position_ids)
print(f"\nStep 3: Position Embedding")
print(f"  位置编码表大小: {list(gpt2_model.wpe.weight.shape)}  [max_positions=1024, d_model=768]")
print(f"  输出形状: {list(position_embeds.shape)}")

# Token Embedding + Position Embedding = 最终输入
hidden_states = token_embeds + position_embeds
print(f"\n  Token Embed + Position Embed = 初始 Hidden States")
print(f"  形状: {list(hidden_states.shape)}")

In [ ]:
# =============================================
# 5.2 追踪每一层 Transformer Block 的输出
# =============================================
# 让数据逐层通过 GPT-2 的 12 个 Block，记录每层输出的统计信息

print("Step 4-15: 经过 12 层 Transformer Block")
print("=" * 60)
print(f"{'层':>4}  {'输出形状':<25} {'均值':>10} {'标准差':>10} {'最大值':>10}")
print("-" * 60)

h = hidden_states.clone().detach()  # 从初始 hidden states 开始

# gpt2_model.h 是一个 ModuleList，包含 12 个 Transformer Block
for i, block in enumerate(gpt2_model.h):
    # 每个 block 返回一个 tuple，第一个元素是 hidden states
    with torch.no_grad():  # 不计算梯度，节省内存
        outputs = block(h)
        h = outputs[0]  # 取出 hidden states
    
    # 打印每层的统计信息
    print(f"  {i:>2}  {str(list(h.shape)):<25} "
          f"{h.mean().item():>10.4f} "
          f"{h.std().item():>10.4f} "
          f"{h.abs().max().item():>10.4f}")

# 最终 LayerNorm
# gpt2_model.ln_f: GPT-2 最后的 LayerNorm 层
h = gpt2_model.ln_f(h)
print(f"\nStep 16: 最终 LayerNorm")
print(f"  形状: {list(h.shape)}")
print(f"  均值: {h.mean().item():.4f}, 标准差: {h.std().item():.4f}")

print(f"\n关键观察:")
print(f"  1. 每层输出形状都是 {list(h.shape)} — Block 不改变形状！")
print(f"  2. 经过每一层，隐藏状态的统计特性会变化 — 模型在逐步处理信息")
print(f"  3. 最终 LayerNorm 让输出标准化 — 稳定后续计算")

In [ ]:
# =============================================
# 5.3 打印每一步的形状变化总结
# =============================================

print(f"\n{'=' * 60}")
print(f"数据流过 GPT-2 的完整形状变化")
print(f"{'=' * 60}")
print(f"")
print(f"  输入文本:       \"{sentence}\"")
print(f"  Tokenizer:      [{seq_len}] token IDs")
print(f"  Token Embed:    [1, {seq_len}, 768]  ← 每个 token 变成 768 维向量")
print(f"  + Pos Embed:    [1, {seq_len}, 768]  ← 加入位置信息")
print(f"  Block 0:        [1, {seq_len}, 768]  ← 形状不变!")
print(f"  Block 1:        [1, {seq_len}, 768]")
print(f"  ...")
print(f"  Block 11:       [1, {seq_len}, 768]")
print(f"  LayerNorm:      [1, {seq_len}, 768]")
print(f"  LM Head:        [1, {seq_len}, 50257]  ← 映射到词表大小")
print(f"")
print(f"  最终输出: 每个位置给出词表中 50257 个词各自的概率")
print(f"  预测下一个词: 取最后一个位置概率最高的词")

---
## 6. 为什么 Transformer 这么强？

Transformer 在 2017 年提出后迅速取代了 RNN/LSTM，成为 NLP 的标准架构。原因有三个：

### 优势 1：训练时的并行计算

```
RNN 训练: 必须等上一步算完才能算下一步（串行，GPU 大量空闲）

  "I" → [RNN] → "love" → [RNN] → "deep" → [RNN] → "learning" → [RNN]
   等等等等      等等等等        等等等等          等等等等

Transformer 训练: 所有位置同时计算（并行，GPU 利用率拉满）

  "I"        → [Attention] ─┐
  "love"     → [Attention] ─┤ 同时进行!
  "deep"     → [Attention] ─┤ GPU 全速运转
  "learning" → [Attention] ─┘
```

> **注意**: 这里说的是**训练时的并行**。单条序列推理时，Transformer 的 O(n²) 注意力反而比 RNN 的 O(n) 慢。但训练需要处理海量数据，并行能力决定了谁能训练更大的模型。

### 优势 2：长距离依赖

在 RNN 中，"第 1 个词"到"第 100 个词"的信息需要传递 99 步，信息会逐步衰减。

在 Transformer 中，任意两个词通过注意力**直接连接**，路径长度为 O(1)。

### 优势 3：可扩展性（Scalability）

Transformer 的性能随着模型大小、数据量和计算量的增加而**可预测地提升**（Scaling Law）。这让 GPT-3（175B 参数）→ GPT-4 这样的大规模训练成为可能。

![为什么 Transformer 这么强](images/why_transformer.png)

*图5：Transformer 的三大优势——训练并行、长距离依赖、可扩展性*

In [ ]:
# =============================================
# 6.1 对比 RNN vs Transformer 的计算时间
# =============================================
# 这个实验揭示 RNN 和 Transformer 各自的计算特性：
# - RNN: O(n) 时间复杂度（每步串行，但每步计算量小）
# - Transformer: O(n²) 时间复杂度（Self-Attention 计算所有词对的关系）
#
# 重要理解：Transformer 的优势不是"单次推理更快"，而是：
# 1. 训练时可以并行（RNN 必须串行，GPU 利用率低）
# 2. 长距离依赖不会衰减（RNN 第100步几乎忘了第1步）
# 3. 规模效应好（Scaling Law）

d_model = 128

# 简单的 RNN 层（用于对比）
rnn = nn.RNN(input_size=d_model, hidden_size=d_model, batch_first=True)

# 简单的 Transformer Block（我们前面实现的）
transformer_block = TransformerBlock(d_model=d_model, n_heads=4)

# 测试不同序列长度，包括较长的序列来体现 O(n²) 的增长
seq_lengths = [16, 32, 64, 128, 256, 512, 1024, 2048]
rnn_times = []
transformer_times = []

for seq_len in seq_lengths:
    x = torch.randn(1, seq_len, d_model)
    
    # 测量 RNN 时间
    # 多次运行取平均，减少噪声
    n_runs = 20
    start = time.time()
    for _ in range(n_runs):
        with torch.no_grad():
            rnn(x)
    rnn_time = (time.time() - start) / n_runs * 1000  # 转换为毫秒
    rnn_times.append(rnn_time)
    
    # 测量 Transformer 时间
    start = time.time()
    for _ in range(n_runs):
        with torch.no_grad():
            transformer_block(x)
    transformer_time = (time.time() - start) / n_runs * 1000
    transformer_times.append(transformer_time)

# 打印结果
print(f"{'Seq Length':>12} {'RNN (ms)':>12} {'Transformer (ms)':>18} {'Growth':>12}")
print("-" * 58)
for i, (sl, rt, tt) in enumerate(zip(seq_lengths, rnn_times, transformer_times)):
    # 计算相对于前一次的增长倍数，展示复杂度差异
    if i > 0:
        rnn_growth = rt / rnn_times[i-1]
        tf_growth = tt / transformer_times[i-1]
        growth_str = f"RNN {rnn_growth:.1f}x, TF {tf_growth:.1f}x"
    else:
        growth_str = "-"
    print(f"{sl:>12} {rt:>12.2f} {tt:>18.2f} {growth_str:>12}")

print()
print("观察：序列长度翻倍时...")
print("  - RNN 耗时约 ×2（线性增长 O(n)）")
print("  - Transformer 耗时约 ×4（二次增长 O(n²)，因为 Attention 计算所有词对）")
print()
print("那为什么 Transformer 还能取代 RNN？答案在下面的训练并行度对比中 ↓")

In [ ]:
# =============================================
# 6.2 可视化计算时间 + 训练并行度对比
# =============================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图: 推理时间对比 ---
axes[0].plot(seq_lengths, rnn_times, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='RNN — O(n)')
axes[0].plot(seq_lengths, transformer_times, 's-', color='#3498db', linewidth=2, markersize=8, label='Transformer — O(n²)')
axes[0].set_xlabel('Sequence Length', fontsize=13)
axes[0].set_ylabel('Time (ms)', fontsize=13)
axes[0].set_title('Single Sequence Forward Pass (CPU)', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# --- 右图: 训练并行度对比（概念示意） ---
# RNN 训练时必须串行处理 n 个时间步，GPU 并行度 = 1（沿时间轴）
# Transformer 训练时所有位置同时计算，GPU 并行度 = n
# 这才是 Transformer 取代 RNN 的核心原因
parallel_levels = [1] * len(seq_lengths)  # RNN: 并行度始终为 1（串行）
axes[1].bar(
    [x - 0.2 for x in range(len(seq_lengths))], parallel_levels,
    width=0.35, color='#e74c3c', label='RNN (serial: 1 step at a time)', alpha=0.8
)
axes[1].bar(
    [x + 0.2 for x in range(len(seq_lengths))], seq_lengths,
    width=0.35, color='#3498db', label='Transformer (parallel: all at once)', alpha=0.8
)
axes[1].set_xticks(range(len(seq_lengths)))
axes[1].set_xticklabels(seq_lengths)
axes[1].set_xlabel('Sequence Length', fontsize=13)
axes[1].set_ylabel('Parallelism (positions computed simultaneously)', fontsize=13)
axes[1].set_title('Training Parallelism: Why Transformer Wins', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("核心理解:")
print("  左图 — 单条序列推理: Transformer O(n²) 其实比 RNN O(n) 慢!")
print("         RNN 每步只做一次矩阵乘法, Transformer 要算 n×n 的注意力矩阵")
print()
print("  右图 — 训练并行度: 这才是 Transformer 的杀手锏!")
print("         RNN 必须一步步串行计算（第2步等第1步完成），GPU 大量空闲")
print("         Transformer 所有位置同时计算，GPU 利用率拉满")
print("         → 序列长 2048 时: Transformer 并行度是 RNN 的 2048 倍!")
print()
print("  结论: Transformer 赢在训练效率，而非单次推理速度")
print("        这就是为什么 Transformer 能训练出 GPT-4 这样的超大模型")

---
## 7. 各组件参数量分析

了解参数量分布能帮助你理解：模型的"大脑"主要在哪里？哪些组件最消耗资源？

In [ ]:
# =============================================
# 7.1 GPT-2 各组件参数量详细分析
# =============================================

def detailed_param_breakdown(model, model_name):
    """对模型的每个参数进行分类统计
    
    model.named_parameters() 返回 (名称, 参数) 的迭代器
    名称格式: 'h.0.attn.c_attn.weight' 表示第0层的Attention的权重
    """
    categories = {}
    
    for name, param in model.named_parameters():
        n = param.numel()
        name_lower = name.lower()
        
        # 根据参数名称分类
        if 'wte' in name_lower or ('embed' in name_lower and 'position' not in name_lower):
            cat = 'Token Embedding'
        elif 'wpe' in name_lower or 'position' in name_lower:
            cat = 'Position Embedding'
        elif 'attn' in name_lower or 'attention' in name_lower:
            cat = 'Attention (Q/K/V/O)'
        elif 'mlp' in name_lower or 'intermediate' in name_lower:
            cat = 'FFN'
        elif 'ln' in name_lower or 'norm' in name_lower or 'layernorm' in name_lower:
            cat = 'LayerNorm'
        elif 'pooler' in name_lower:
            cat = 'Pooler'
        else:
            cat = 'Other'
        
        categories[cat] = categories.get(cat, 0) + n
    
    return categories


gpt2_breakdown = detailed_param_breakdown(gpt2_model, 'GPT-2')
total = sum(gpt2_breakdown.values())

print("GPT-2 参数量分布")
print("=" * 55)
print(f"{'组件':<25} {'参数量':>12} {'占比':>8}")
print("-" * 55)
# sorted: 按参数量从大到小排序
for cat, count in sorted(gpt2_breakdown.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    # 用 █ 字符画简单的柱状图
    bar = '█' * int(pct / 2)
    print(f"  {cat:<23} {count:>12,} {pct:>7.1f}% {bar}")
print("-" * 55)
print(f"  {'Total':<23} {total:>12,}")

In [ ]:
# =============================================
# 7.2 参数量可视化（柱状图）
# =============================================

# 按参数量排序
sorted_components = sorted(gpt2_breakdown.items(), key=lambda x: -x[1])
names = [c[0] for c in sorted_components]
counts = [c[1] for c in sorted_components]
total = sum(counts)

# 计算百分比
percentages = [c / total * 100 for c in counts]

# 配色方案
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 柱状图
bars = axes[0].barh(range(len(names)), counts, color=colors[:len(names)], edgecolor='white')
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=11)
axes[0].set_xlabel('Parameter Count', fontsize=12)
axes[0].set_title('GPT-2 Parameter Distribution by Component', fontsize=13)
# 在每个柱子右边标注具体数值
for i, (count, pct) in enumerate(zip(counts, percentages)):
    axes[0].text(count + total * 0.01, i, f'{count/1e6:.1f}M ({pct:.1f}%)',
                va='center', fontsize=10)
axes[0].invert_yaxis()  # 最大的在上面

# 右图: 饼图
# autopct: 自动显示百分比, startangle: 起始角度
wedges, texts, autotexts = axes[1].pie(
    counts, labels=names, autopct='%1.1f%%',
    colors=colors[:len(names)], startangle=90,
    textprops={'fontsize': 10}
)
axes[1].set_title('GPT-2 Parameter Share', fontsize=13)

plt.tight_layout()
plt.show()

print("关键发现:")
# 找到占比最大的两个组件
top2 = sorted_components[:2]
print(f"  1. {top2[0][0]} 和 {top2[1][0]} 占了绝大部分参数")
print(f"  2. LayerNorm 的参数量几乎可以忽略不计")
print(f"  3. 这就是为什么 Transformer 的计算主要在 Attention 和 FFN 上")
print(f"  4. 想让模型更大？主要增加 Attention 和 FFN 的参数（增加 d_model 或层数）")

---
## 8. 本章总结

### 6 个关键要点

**1. Transformer 的整体结构** — 输入 → Embedding + 位置编码 → N 个 Transformer Block → 输出。像一条流水线，每个 Block 结构相同但参数不同

**2. Transformer Block 的四大组件** — Multi-Head Attention（收集上下文）→ Add & LayerNorm（稳定训练）→ FFN（独立处理信息）→ Add & LayerNorm。残差连接保证信息不丢失

**3. Encoder vs Decoder** — 结构几乎一样，唯一区别是 Decoder 使用因果 mask 防止看到未来的词。Encoder 双向看，Decoder 单向看

**4. 三种变体** — Encoder-only（BERT，理解）、Decoder-only（GPT，生成）、Encoder-Decoder（T5，转换）。现代大模型主要用 Decoder-only

**5. 数据流的形状不变性** — Transformer Block 的输入和输出形状完全一致 [batch, seq_len, d_model]，所以可以像乐高积木一样无限堆叠

**6. Transformer 的三大优势** — 并行计算（比 RNN 快）、长距离依赖（O(1) 路径）、可扩展性（Scaling Law）

### 下一章预告

在第7章中，我们将学习 **位置编码（Positional Encoding）**：
- 为什么 Transformer 需要位置编码？（注意力机制本身不区分词的顺序）
- 正弦位置编码 vs 可学习位置编码
- RoPE（旋转位置编码）—— 现代大模型的选择

---

> **参考资料**
> - Vaswani et al., 2017: *Attention Is All You Need* — Transformer 原始论文
> - Jay Alammar: *The Illustrated Transformer* — Transformer 可视化讲解
> - Devlin et al., 2019: *BERT: Pre-training of Deep Bidirectional Transformers* — BERT 论文
> - Radford et al., 2019: *Language Models are Unsupervised Multitask Learners* — GPT-2 论文
> - Kaplan et al., 2020: *Scaling Laws for Neural Language Models* — Scaling Law 论文